LIBRARAIES 


In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import os
import time
from sklearn.preprocessing import LabelEncoder

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")



The code below cleans and prepares your Roman Urdu emotion dataset for machine learning. 
It reads five separate emotion files (joy, sadness, sarcasm, anger, neutral), cleans the text by removing garbage characters and extra spaces, labels each sentence with its corresponding emotion, splits the data into 80% training and 20% testing sets while maintaining the emotion distribution, and finally saves everything as organized CSV files for model training.

In [ ]:
# Data Finalization 
import pandas as pd
import numpy as np
import os
import re
from sklearn.model_selection import train_test_split

# -------------------------------
# CONFIG: FILES AND LABELS
# -------------------------------
file_config = {
    '/kaggle/input/jazbatai-data/training_data_joy.txt': 'joy',
    '/kaggle/input/jazbatai-data/training_data_sadness.txt': 'sadness', 
    '/kaggle/input/jazbatai-data/training_data_sarcasam.txt': 'sarcasm',
    '/kaggle/input/jazbatai-data/training_data_anger.txt': 'anger',
    '/kaggle/input/jazbatai-data/training_data_neutral.txt': 'neutral'
}

# -------------------------------
# CREATE OUTPUT FOLDER
# -------------------------------
output_folder = '/kaggle/input/jazbatai-data/labelled-data/'

# Make folder if it doesn't exist
output_folder = '/kaggle/working/labelled-data/'
os.makedirs(output_folder, exist_ok=True)


# -------------------------------
# PROCESS FILES
# -------------------------------
for file_path, emotion_label in file_config.items():
    try:
        cleaned_lines = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f.readlines():
                text = line.strip()
                if not text:
                    continue  # skip empty lines

                # --- Remove Unicode & garbage text ---
                text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Non-ASCII
                text = re.sub(r'ï¿½+', ' ', text)  # specific garbage
                text = re.sub(r'[ ]+', ' ', text)  # multiple spaces -> single space
                text = text.strip()

                if not text:
                    continue  # skip empty after cleaning

                cleaned_lines.append(text)

        print(f"🎭 {emotion_label}: {len(cleaned_lines)} sentences after cleaning")

        if not cleaned_lines:
            continue

        df = pd.DataFrame({
            'text': cleaned_lines,
            'label': emotion_label
        })

        # Split 80/20 per emotion
        train_df, test_df = train_test_split(
            df,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )

        all_training_data.append(train_df)
        all_testing_data.append(test_df)

        print(f"   Training: {len(train_df)}, Testing: {len(test_df)}")

    except Exception as e:
        print(f"    Error processing {file_path}: {e}")

# -------------------------------
# COMBINE DATA
# -------------------------------
final_training_df = pd.concat(all_training_data, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
final_testing_df = pd.concat(all_testing_data, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Full dataset
full_df = pd.concat([final_training_df, final_testing_df], ignore_index=True)

print("\n FINAL DATASET STATISTICS")
print(f"Training: {len(final_training_df)} samples")
print(f"Testing:  {len(final_testing_df)} samples")
print(f"Full dataset: {len(full_df)} samples")

# -------------------------------
# SAVE CSV FILES IN labelled-data
# -------------------------------
training_csv_path = os.path.join(output_folder, 'training.csv')
testing_csv_path = os.path.join(output_folder, 'testing.csv')
full_csv_path    = os.path.join(output_folder, 'full_dataset.csv')

final_training_df.to_csv(training_csv_path, index=False, encoding='utf-8')
final_testing_df.to_csv(testing_csv_path, index=False, encoding='utf-8')
full_df.to_csv(full_csv_path, index=False, encoding='utf-8')

print(f"\n CSV files saved in {output_folder}")
print(f"Training: {training_csv_path}")
print(f"Testing:  {testing_csv_path}")
print(f"Full dataset: {full_csv_path}")

# -------------------------------
# OPTIONAL: LABEL DISTRIBUTION
# -------------------------------
print("\n LABEL DISTRIBUTION IN TRAINING DATA")
print(final_training_df['label'].value_counts())

print("\n LABEL DISTRIBUTION IN TESTING DATA")
print(final_testing_df['label'].value_counts())


The code below performs light text cleaning on your Roman Urdu dataset by removing URLs, usernames, and excessive punctuation while preserving the original text structure, then saves the cleaned versions for model training.

In [ ]:
# MINIMAL PREPROCESSING 

import pandas as pd
import re

# -----------------------------
# 1) Preprocessing Function
# -----------------------------
def preprocess_roman_urdu(text):
    """
    Minimal preprocessing for Roman Urdu text for BERT embeddings.
    """
    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove usernames
    text = re.sub(r'@\w+', '', text)

    # Reduce repeated punctuation (!!! → !, ??? → ?)
    text = re.sub(r'([!?]){2,}', r'\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# -----------------------------
# 2) Load Input CSVs
# -----------------------------
train_path = "/kaggle/input/jazbatai-dataset/training.csv"
test_path  = "/kaggle/input/jazbatai-dataset/testing.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Before preprocessing:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# -----------------------------
# 3) Apply preprocessing
# -----------------------------
train_df["text"] = train_df["text"].apply(preprocess_roman_urdu)
test_df["text"] = test_df["text"].apply(preprocess_roman_urdu)


# -----------------------------
# 4) Save Output Clean CSVs
# -----------------------------
output_train = "/kaggle/working/training_clean.csv"
output_test  = "/kaggle/working/testing_clean.csv"

train_df.to_csv(output_train, index=False)
test_df.to_csv(output_test, index=False)

print("\nAfter preprocessing:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nSample cleaned data:")
print(train_df.head())
print("\nFiles saved to:")
print(output_train)
print(output_test)


Naive Bayes Pipeline Execution:

Loads & Vectorizes: Reads cleaned Roman Urdu text and converts it to numerical features using TF-IDF with 1-2 grams

Trains & Predicts: Fits a Multinomial Naive Bayes classifier and generates predictions on test data

Measures Performance: Calculates accuracy, F1 scores,

Visualizes Results: Shows confusion matrix and per-class F1 scores with performance status labels

In [ ]:
# ============================================================
# NAIVE BAYES (TF-IDF + MultinomialNB)
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import seaborn as sns

print("Starting Naive Bayes Pipeline...")

# -----------------------------
# LOAD DATA
# -----------------------------
TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset/testing_clean.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()

y_train = df_train["label"].astype(str).values
y_test  = df_test["label"].astype(str).values

print(f"Train samples: {len(train_texts)}")
print(f"Test samples :  {len(test_texts)}")

# -----------------------------
# TF-IDF + NAIVE BAYES 
# -----------------------------
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_texts)
X_test  = vectorizer.transform(test_texts)

clf = MultinomialNB()
clf.fit(X_train, y_train)

pred = clf.predict(X_test)

# -----------------------------
# METRICS
# -----------------------------
accuracy = accuracy_score(y_test, pred)
macro_f1 = f1_score(y_test, pred, average="macro")
full_report = classification_report(y_test, pred, output_dict=True)

print("\n" + "="*50)
print("              PERFORMANCE RESULTS")
print("="*50)
print(f"Accuracy                    : {accuracy:.4f}")
print(f"Macro F1                    : {macro_f1:.4f}")
print(f"Total Samples Processed     : {len(train_texts) + len(test_texts)}")
print("\nClassification Report:")
print(classification_report(y_test, pred))

# -----------------------------
# CONFUSION MATRIX
# -----------------------------
labels = sorted(list(set(y_train)))

cm = confusion_matrix(y_test, pred, labels=labels)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.title("Confusion Matrix - Naive Bayes (5 Emotions)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ============================================================
# F1 SCORE BAR GRAPH PER CLASS
# ============================================================

class_f1_scores = {label: full_report[label]["f1-score"] for label in labels}

plt.figure(figsize=(7, 5))
plt.bar(class_f1_scores.keys(), class_f1_scores.values())
plt.title("F1 Score per Class")
plt.xlabel("Emotion Class")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.show()

# ============================================================
# PERFORMANCE TABLE
# ============================================================

rows = []
for label in labels:
    precision = full_report[label]["precision"]
    recall = full_report[label]["recall"]
    f1 = full_report[label]["f1-score"]

    if f1 >= 0.85:
        status = "High Performer"
    elif f1 >= 0.70:
        status = "Moderate"
    else:
        status = "Mixed/Weak"

    rows.append([label, precision, recall, f1, status])

performance_df = pd.DataFrame(
    rows,
    columns=["Class", "Precision", "Recall", "F1-Score", "Status"]
)

print("\nPerformance Table:")
print(performance_df.to_string(index=False))

print("\nDone.")


mBERT EMBEDDING 

Here's a step-by-step breakdown of the mBERT embedding generation process:

 Step 1: Setup & Installation
- Installs Hugging Face transformers library
- Imports necessary PyTorch and NLP modules

 Step 2: Configuration
- Sets paths for local mBERT model and dataset
- Defines parameters: max sequence length (64), batch size (32)
- Detects available device (GPU/CPU)

 Step 3: Model Loading
- Loads pre-trained multilingual BERT tokenizer and model from local files
- Moves model to GPU if available
- Sets model to evaluation mode (no training)

 Step 4: Pooling Strategies
Defines four different ways to convert token embeddings to sentence embeddings:
- Mean Pooling: Average of all token embeddings
- Mean Squared Pooling: Average of squared token embeddings  
- Max Pooling: Maximum value across tokens
- CLS Pooling: Uses the special [CLS] token embedding
- Multi Pooling: Combines mean, max, and CLS embeddings

 Step 5: Batch Processing
- Processes texts in batches for memory efficiency
- Tokenizes texts with padding/truncation to 64 tokens
- Generates embeddings using all pooling strategies
- Converts results to NumPy arrays and stores in CPU memory

 Step 6: Data Loading & Execution
- Loads cleaned training and test CSV files
- Converts text columns to lists
- Runs embedding generation on both datasets

 Step 7: Output Saving
Saves three types of embeddings for both train/test:
- Mean-squared pooled embeddings
- Combined multi-pooled embeddings  
- CLS token embeddings
- All saved as .npy files for downstream ML tasks

Final Output: 6 NumPy files containing rich contextual embeddings ready for classical ML model training.

In [ ]:
#GENERATING M-BERT EMBEDDINGS !


# ============================================================
# INSTALL
# ============================================================
!pip install -q transformers sentencepiece

# ============================================================
# IMPORTS
# ============================================================
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import os

# ============================================================
# CONFIG
# ============================================================
MODEL_PATH = "/kaggle/input/mbert-model"   #  LOCAL MBERT FOLDER
MAX_LEN = 64
BATCH_SIZE = 32

TRAIN_PATH = "/kaggle/input/your-dataset/training_clean.csv"
TEST_PATH  = "/kaggle/input/your-dataset/testing_clean.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Check files to confirm
print("Local mBERT files:", os.listdir(MODEL_PATH))

# ============================================================
# LOAD LOCAL MODEL + TOKENIZER
# ============================================================
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertModel.from_pretrained(MODEL_PATH)
model.to(DEVICE)
model.eval()

# ============================================================
# POOLING FUNCTIONS
# ============================================================
def mean_squared_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    squared = hidden_states  2
    masked = squared * mask
    summed = torch.sum(masked, dim=1)
    count = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / count

def mean_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    masked = hidden_states * mask
    summed = torch.sum(masked, dim=1)
    count = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / count

def max_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    masked = hidden_states.masked_fill(mask == 0, -1e9)
    return torch.max(masked, dim=1).values

def cls_pooling(hidden_states):
    return hidden_states[:, 0, :]

# ============================================================
# BATCH EMBEDDING FUNCTION
# ============================================================
def create_embeddings_batch(texts, batch_size=32):
    emb_ms = []
    emb_multi = []
    emb_cls = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Batches"):
            try:
                batch_texts = texts[i:i+batch_size]

                if not batch_texts:
                    continue

                inputs = tokenizer(
                    batch_texts,
                    padding="max_length",
                    truncation=True,
                    max_length=MAX_LEN,
                    return_tensors="pt"
                ).to(DEVICE)

                outputs = model(inputs)
                hidden = outputs.last_hidden_state

                ms = mean_squared_pooling(hidden, inputs['attention_mask'])
                m  = mean_pooling(hidden, inputs['attention_mask'])
                mx = max_pooling(hidden, inputs['attention_mask'])
                c  = cls_pooling(hidden)

                multi = torch.cat([m, mx, c], dim=1)

                emb_ms.extend(ms.cpu().numpy())
                emb_multi.extend(multi.cpu().numpy())
                emb_cls.extend(c.cpu().numpy())

            except Exception as e:
                print(f" Error in batch {i}: {e}")
                continue

    return (
        np.array(emb_ms),
        np.array(emb_multi),
        np.array(emb_cls),
    )

# ============================================================
# LOAD DATA
# ============================================================
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

train_texts = df_train["text"].astype(str).tolist()
test_texts  = df_test["text"].astype(str).tolist()

print("Train samples:", len(train_texts))
print("Test samples:", len(test_texts))

# ============================================================
# GENERATE EMBEDDINGS
# ============================================================
train_ms, train_multi, train_cls = create_embeddings_batch(train_texts)
test_ms,  test_multi,  test_cls  = create_embeddings_batch(test_texts)

print("\nShapes:")
print("Mean-Squared:", train_ms.shape, test_ms.shape)
print("Multi:", train_multi.shape, test_multi.shape)
print("CLS:", train_cls.shape, test_cls.shape)

# ============================================================
# SAVE OUTPUT
# ============================================================
np.save("mbert_train_mean_squared.npy", train_ms)
np.save("mbert_test_mean_squared.npy", test_ms)

np.save("mbert_train_mean_max_cls.npy", train_multi)
np.save("mbert_test_mean_max_cls.npy", test_multi)

np.save("mbert_train_cls.npy", train_cls)
np.save("mbert_test_cls.npy", test_cls)



 SVM Classification Pipeline Breakdown:

 Step 1: Setup & Data Loading
- Loads three types of mBERT embeddings (mean_squared, cls, multi)
- Reads original CSV files for emotion labels
- Defines emotion class names for visualization

 Step 2: Visualization Utilities
- Creates `fig_to_html()` function to convert matplotlib plots to base64 images
- Enables embedding charts directly in HTML reports

 Step 3: SVM Training & Evaluation
For each embedding type:
- Linear SVM: Uses LinearSVC with C=0.5, optimized for high-dimensional data
- RBF SVM: Uses SVC with RBF kernel, C=2 for non-linear classification
- Preprocessing: StandardScaler normalizes embeddings for better SVM performance
- Metrics: Calculates accuracy, macro F1, per-class F1 scores

 Step 4: Visualization Generation
- Confusion Matrix: Shows prediction patterns across 5 emotion classes
- F1 Bar Charts: Visualizes per-class performance for quick comparison
- Both visualizations converted to HTML images

Output: Single HTML file with comprehensive SVM performance comparison across different embedding strategies for Roman Urdu emotion classification.

In [ ]:
# SVM AS CLASSIFICATION HEADS FOR THE EMBEDDINGS 

# =============================================================
# IMPORTS
# =============================================================
import numpy as np
import pandas as pd
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from io import StringIO
import base64

# =============================================================
# LOAD EMBEDDINGS
# =============================================================
embeddings = {
    "mean_squared": {
        "train": np.load("mbert_train_mean_squared.npy"),
        "test":  np.load("mbert_test_mean_squared.npy")
    },
    "cls": {
        "train": np.load("mbert_train_cls.npy"),
        "test":  np.load("mbert_test_cls.npy")
    },
    "multi": {
        "train": np.load("mbert_train_mean_max_cls.npy"),
        "test":  np.load("mbert_test_mean_max_cls.npy")
    }
}

df_train = pd.read_csv("/kaggle/input/your-dataset/training_clean.csv")
df_test  = pd.read_csv("/kaggle/input/your-dataset/testing_clean.csv")

y_train = df_train["label"]
y_test  = df_test["label"]

label_names = ["ANGER", "JOY", "NEUTRAL", "SADNESS", "SARCASM"]

# =============================================================
# HELPERS — Convert matplotlib image → HTML <img>
# =============================================================
def fig_to_html():
    tmpfile = StringIO()
    plt.savefig("tmp.png", bbox_inches='tight')
    plt.close()
    with open("tmp.png", "rb") as f:
        encoded = base64.b64encode(f.read()).decode()
    return f'<img src="data:image/png;base64,{encoded}" style="width:400px;">'

# =============================================================
# TRAIN + EVALUATE SVM
# =============================================================
def evaluate_svm(X_train, X_test, y_train, y_test, kernel_name):
    if kernel_name == "linear":
    model = LinearSVC(
        C=0.5,
        max_iter=30000,
        dual=False  # recommended when n_samples > n_features
    )

    else:
        model = SVC(kernel="rbf", C=2, gamma="scale")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    report = classification_report(y_test, preds, target_names=label_names)

    # --- Confusion matrix ---
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_names, yticklabels=label_names)
    plt.title(f"Confusion Matrix — {kernel_name.upper()}")
    cm_html = fig_to_html()

    # --- F1 Histogram ---
    f1_scores = f1_score(y_test, preds, average=None)
    plt.figure(figsize=(6,3))
    plt.bar(label_names, f1_scores)
    plt.title(f"F1 per Class — {kernel_name.upper()}")
    plt.xticks(rotation=45)
    f1_html = fig_to_html()

    return acc, f1, report, cm_html, f1_html


# =============================================================
# GENERATE ONE BIG HTML FILE
# =============================================================
html_output = "<h1>SVM Results (All Embeddings × Both Kernels)</h1>"

for embed_name, data in embeddings.items():
    X_train = data["train"]
    X_test  = data["test"]

    html_output += f"<hr><h2>Embedding Type: {embed_name.upper()}</h2>"

    # ------- Linear SVM -------
    acc, f1, report, cm_img, f1_img = evaluate_svm(
        X_train, X_test, y_train, y_test, kernel_name="linear"
    )
    html_output += f"""
    <h3>Linear SVM</h3>
    <b>Accuracy:</b> {acc:.4f}<br>
    <b>Macro F1:</b> {f1:.4f}<br>
    <pre>{report}</pre>
    <h4>Confusion Matrix</h4>
    {cm_img}
    <h4>F1 Scores</h4>
    {f1_img}
    """

    # ------- RBF SVM -------
    acc, f1, report, cm_img, f1_img = evaluate_svm(
        X_train, X_test, y_train, y_test, kernel_name="rbf"
    )
    html_output += f"""
    <h3>RBF SVM</h3>
    <b>Accuracy:</b> {acc:.4f}<br>
    <b>Macro F1:</b> {f1:.4f}<br>
    <pre>{report}</pre>
    <h4>Confusion Matrix</h4>
    {cm_img}
    <h4>F1 Scores</h4>
    {f1_img}
    """

# =============================================================
# SAVE HTML
# =============================================================
with open("svm_all_results.html", "w", encoding="utf-8") as f:
    f.write(html_output)

print(" HTML file generated: svm_all_results.html")
